# 01 — Layer 1: Bronze Tier

🔧 Script

Per-source adapters write raw, immutable bronze records. Every source is treated as a black box at this layer — no format conversion, no merging. This notebook tours what the bronze tier looks like.


## 1. Source inventory

In [ ]:
from pathlib import Path
import json

ATLAS_ROOT = Path("..").resolve()
CORPUS     = ATLAS_ROOT / "corpus"

sources = sorted((CORPUS / "bronze").iterdir())
for s in sources:
    if not s.is_dir():
        continue
    patients = [p.name for p in s.iterdir() if p.is_dir()]
    print(f"  bronze/{s.name}/  → patients: {patients}")


## 2. Per-source metadata.json — the SourceMetadata contract

In [ ]:
# 🔧 Script — every adapter writes metadata.json alongside the data file
meta = json.loads((CORPUS / "bronze" / "synthea" / "rhett759" / "metadata.json").read_text())
for k, v in meta.items():
    print(f"  {k}: {v}")


## 3. SyntheaAdapter — listing patients and ingesting

In [ ]:
# 🔧 Script
from ehi_atlas.adapters.synthea import SyntheaAdapter

adapter = SyntheaAdapter(
    source_root=CORPUS / "_sources" / "synthea" / "raw",
    bronze_root=CORPUS / "bronze",
)
patients = adapter.list_patients()
print("Available patients:", patients)


In [ ]:
# Run ingest (idempotent — same output every time due to frozen ACQUISITION_TS)
meta = adapter.ingest("rhett759")
print("source:       ", meta.source)
print("patient_id:   ", meta.patient_id)
print("document_type:", meta.document_type)
print("fetched_at:   ", meta.fetched_at)
print("sha256:       ", meta.sha256[:16], "...")


## 4. Inspect the bronze FHIR Bundle

In [ ]:
import json

bronze_bundle = json.loads(
    (CORPUS / "bronze" / "synthea" / "rhett759" / "data.json").read_text()
)
entries = bronze_bundle.get("entry", [])
print(f"Total entries: {len(entries)}")

# Count by resource type
from collections import Counter
types = Counter(e["resource"]["resourceType"] for e in entries if "resource" in e)
for rtype, count in sorted(types.items()):
    print(f"  {rtype}: {count}")


## 5. Compare bronze shapes across sources

In [ ]:
# 🔧 Script — same contract, different data shapes
sources_info = {}
for source in ["synthea", "epic-ehi", "lab-pdf", "synthesized-clinical-note", "synthea-payer"]:
    patient_dir = CORPUS / "bronze" / source / "rhett759"
    if not patient_dir.exists():
        sources_info[source] = "not present"
        continue
    files = [f.name for f in sorted(patient_dir.iterdir())]
    sources_info[source] = files

for src, files in sources_info.items():
    print(f"  {src}: {files}")


**Next:** [02_layer2_synthea_standardize.ipynb](./02_layer2_synthea_standardize.ipynb)